# Localization & Classification Pipeline

Pipeline lengkap untuk:
1. Object localization menggunakan OWL-ViT
2. Cropping objek pakaian
3. Klasifikasi Jenis (Traditional ML Ensemble)
4. Klasifikasi Warna (Pretrained Deep Learning Model)

## 1. Import Libraries

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from PIL import Image
import cv2
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from transformers import OwlViTProcessor, OwlViTForObjectDetection
from transformers import AutoFeatureExtractor, AutoModel, AutoModelForImageClassification
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression, RidgeClassifier
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully")

## 2. Configuration

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_dir = 'train/train/'
test_dir = 'test/test/'
output_dir = 'results/'

os.makedirs(output_dir, exist_ok=True)

print(f"Device: {device}")
print(f"Train directory: {train_dir}")
print(f"Test directory: {test_dir}")

## 3. Load Data

In [ ]:
train_df = pd.read_csv('train.csv')

print(f"Total training samples: {len(train_df)}")
print(f"\nDataset preview:")
print(train_df.head())
print(f"\nJenis distribution:")
print(train_df['jenis'].value_counts())
print(f"\nWarna distribution:")
print(train_df['warna'].value_counts())

## 4. Initialize Models

In [ ]:
from transformers import AutoImageProcessor

owl_processor = OwlViTProcessor.from_pretrained("google/owlvit-base-patch32")
owl_model = OwlViTForObjectDetection.from_pretrained("google/owlvit-base-patch32").to(device)
owl_model.eval()

# Use EfficientNet-B0 for feature extraction (lighter and efficient)
# Note: EfficientNet requires AutoImageProcessor, not AutoFeatureExtractor
feature_extractor = AutoImageProcessor.from_pretrained("google/efficientnet-b0")
feature_model = AutoModel.from_pretrained("google/efficientnet-b0").to(device)
feature_model.eval()

text_queries = [["clothing", "shirt", "dress", "garment", "apparel"]]

print("OWL-ViT model loaded")
print("EfficientNet-B0 feature extractor loaded")

## 5. Helper Functions

In [ ]:
def load_image(image_id, image_dir):
    for ext in ['.jpg', '.png', '.jpeg']:
        image_path = os.path.join(image_dir, f"{image_id}{ext}")
        if os.path.exists(image_path):
            return Image.open(image_path).convert('RGB'), image_path
    return None, None

def preprocess_image(image, target_size=(768, 768)):
    return image.resize(target_size, Image.LANCZOS)

def detect_object(image, threshold=0.1):
    inputs = owl_processor(text=text_queries, images=image, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = owl_model(**inputs)
    
    target_sizes = torch.tensor([image.size[::-1]]).to(device)
    results = owl_processor.post_process_object_detection(
        outputs=outputs, threshold=threshold, target_sizes=target_sizes
    )[0]
    
    return results["boxes"].cpu().numpy(), results["scores"].cpu().numpy()

def crop_object(image, boxes, padding=10):
    if len(boxes) == 0:
        return image
    
    image_np = np.array(image)
    height, width = image_np.shape[:2]
    
    x1, y1, x2, y2 = map(int, boxes[0])
    x1 = max(0, x1 - padding)
    y1 = max(0, y1 - padding)
    x2 = min(width, x2 + padding)
    y2 = min(height, y2 + padding)
    
    return Image.fromarray(image_np[y1:y2, x1:x2])

def extract_features(image):
    """Extract features using EfficientNet-B0"""
    inputs = feature_extractor(images=image, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = feature_model(**inputs)
    
    # EfficientNet outputs: pooler_output or last_hidden_state
    if hasattr(outputs, 'pooler_output') and outputs.pooler_output is not None:
        features = outputs.pooler_output
    elif hasattr(outputs, 'last_hidden_state'):
        # Global average pooling over spatial dimensions
        features = outputs.last_hidden_state.mean(dim=1)
    else:
        # Fallback: take the last hidden state and pool
        features = outputs[0]
        if len(features.shape) == 4:  # [batch, channels, height, width]
            features = features.mean(dim=[2, 3])  # Global average pooling
        elif len(features.shape) == 3:  # [batch, seq_len, hidden_dim]
            features = features.mean(dim=1)
    
    return features.cpu().numpy().flatten()

print("Helper functions defined")

## 6. Extract Features untuk Jenis Classification

In [ ]:
features_list = []
labels_jenis = []

print("Extracting features for Jenis classification...")
for idx, row in train_df.iterrows():
    image, _ = load_image(row['id'], train_dir)
    
    if image is not None:
        preprocessed = preprocess_image(image)
        boxes, scores = detect_object(preprocessed)
        cropped = crop_object(preprocessed, boxes)
        features = extract_features(cropped)
        
        features_list.append(features)
        labels_jenis.append(row['jenis'])
    
    if (idx + 1) % 100 == 0:
        print(f"Processed {idx + 1}/{len(train_df)}")

X_jenis = np.array(features_list)
y_jenis = np.array(labels_jenis)

print(f"\nFeature shape: {X_jenis.shape}")
print(f"Labels: {len(y_jenis)}")

## 7. Train-Test Split untuk Jenis

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Split data
X_train_jenis, X_test_jenis, y_train_jenis, y_test_jenis = train_test_split(
    X_jenis, y_jenis, test_size=0.2, random_state=42, stratify=y_jenis
)

# Scale features
scaler = StandardScaler()
X_train_jenis_scaled = scaler.fit_transform(X_train_jenis)
X_test_jenis_scaled = scaler.transform(X_test_jenis)

print(f"Train set: {X_train_jenis_scaled.shape}")
print(f"Test set: {X_test_jenis_scaled.shape}")

## 8. Compute Class Weights untuk Jenis

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

# Compute class weights
unique_classes_jenis = np.unique(y_train_jenis)
class_weights_jenis = compute_class_weight('balanced', classes=unique_classes_jenis, y=y_train_jenis)
class_weight_dict_jenis = dict(zip(unique_classes_jenis, class_weights_jenis))
sample_weights_jenis = np.array([class_weight_dict_jenis[label] for label in y_train_jenis])

print("Class weights untuk Jenis:")
for cls, weight in class_weight_dict_jenis.items():
    print(f"  {cls}: {weight:.4f}")

## 9. Train Traditional ML Models untuk Jenis

In [ ]:
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

# Initialize models
models_jenis = {
    'XGBoost': XGBClassifier(n_estimators=200, max_depth=7, learning_rate=0.1, random_state=42),
    'HistGradient': HistGradientBoostingClassifier(max_iter=200, max_depth=7, random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=200, max_depth=15, class_weight='balanced', random_state=42),
    'LogisticRegression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'Ridge': RidgeClassifier(alpha=1.0, class_weight='balanced', random_state=42)
}

# Train and evaluate individual models
results_jenis = {}
for name, model in models_jenis.items():
    print(f"\nTraining {name}...")
    
    if name in ['XGBoost', 'HistGradient']:
        model.fit(X_train_jenis_scaled, y_train_jenis, sample_weight=sample_weights_jenis)
    else:
        model.fit(X_train_jenis_scaled, y_train_jenis)
    
    y_pred = model.predict(X_test_jenis_scaled)
    accuracy = accuracy_score(y_test_jenis, y_pred)
    results_jenis[name] = accuracy
    
    print(f"{name} Accuracy: {accuracy:.4f}")

print("\n" + "="*60)
print("Individual Model Results for Jenis:")
for name, acc in results_jenis.items():
    print(f"{name}: {acc:.4f}")

## 10. Create Ensemble Model untuk Jenis

In [ ]:
# Create ensemble
ensemble_jenis = VotingClassifier(
    estimators=list(models_jenis.items()),
    voting='hard'
)

print("Training ensemble model...")
ensemble_jenis.fit(X_train_jenis_scaled, y_train_jenis)

y_pred_ensemble = ensemble_jenis.predict(X_test_jenis_scaled)
ensemble_accuracy = accuracy_score(y_test_jenis, y_pred_ensemble)

print(f"\nEnsemble Accuracy: {ensemble_accuracy:.4f}")
print("\nClassification Report untuk Jenis (Ensemble):")
print(classification_report(y_test_jenis, y_pred_ensemble))

## 11. Confusion Matrix untuk Jenis

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

cm_jenis = confusion_matrix(y_test_jenis, y_pred_ensemble)
plt.figure(figsize=(10, 8))
sns.heatmap(cm_jenis, annot=True, fmt='d', cmap='Blues',
            xticklabels=np.unique(y_jenis),
            yticklabels=np.unique(y_jenis))
plt.title('Confusion Matrix - Jenis Classification (Ensemble)')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

## 12. Multi-Feature Extraction untuk Warna Classification

In [ ]:
def extract_color_histogram(image, bins=32):
    """Extract color histogram features from RGB channels"""
    image_np = np.array(image.resize((224, 224)))
    hist_features = []
    
    for channel in range(3):
        hist = cv2.calcHist([image_np], [channel], None, [bins], [0, 256])
        hist = hist.flatten() / hist.sum()
        hist_features.extend(hist)
    
    return np.array(hist_features)

def extract_hsv_features(image):
    """Extract HSV color moments (mean, std, quartiles)"""
    image_np = np.array(image.resize((224, 224)))
    hsv = cv2.cvtColor(image_np, cv2.COLOR_RGB2HSV)
    
    features = []
    for channel in range(3):
        channel_data = hsv[:, :, channel].flatten()
        features.extend([
            np.mean(channel_data),
            np.std(channel_data),
            np.percentile(channel_data, 25),
            np.percentile(channel_data, 50),
            np.percentile(channel_data, 75)
        ])
    
    return np.array(features)

def extract_lab_features(image):
    """Extract LAB color space features"""
    image_np = np.array(image.resize((224, 224)))
    lab = cv2.cvtColor(image_np, cv2.COLOR_RGB2LAB)
    
    features = []
    for channel in range(3):
        channel_data = lab[:, :, channel].flatten()
        features.extend([
            np.mean(channel_data),
            np.std(channel_data),
            np.percentile(channel_data, 50)
        ])
    
    return np.array(features)

def extract_dominant_colors(image, k=5):
    """Extract dominant colors using K-means"""
    image_np = np.array(image.resize((224, 224)))
    pixels = image_np.reshape(-1, 3).astype(np.float32)
    
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 100, 0.2)
    _, labels, centers = cv2.kmeans(pixels, k, None, criteria, 10, cv2.KMEANS_RANDOM_CENTERS)
    
    unique, counts = np.unique(labels, return_counts=True)
    percentages = counts / counts.sum()
    
    features = np.concatenate([centers.flatten(), percentages])
    
    return features

def extract_combined_color_features(image):
    """Combine all color features"""
    hist_feat = extract_color_histogram(image, bins=32)
    hsv_feat = extract_hsv_features(image)
    lab_feat = extract_lab_features(image)
    dom_feat = extract_dominant_colors(image, k=5)
    
    combined = np.concatenate([hist_feat, hsv_feat, lab_feat, dom_feat])
    
    return combined

print("Color feature extraction functions defined")
print("Features include: RGB Histogram, HSV Moments, LAB Stats, Dominant Colors")

## 13. Extract Multi-Modal Features untuk Warna

In [ ]:
print("Extracting multi-modal color features...")
color_features_list = []
labels_warna = []
deep_features_list = []

for idx, row in train_df.iterrows():
    image, _ = load_image(row['id'], train_dir)
    
    if image is not None:
        preprocessed = preprocess_image(image)
        boxes, scores = detect_object(preprocessed)
        cropped = crop_object(preprocessed, boxes)
        
        # Extract handcrafted color features
        color_features = extract_combined_color_features(cropped)
        color_features_list.append(color_features)
        
        # Extract deep features from EfficientNet-B0 backbone
        deep_features = extract_features(cropped)
        deep_features_list.append(deep_features)
        
        labels_warna.append(row['warna'])
    
    if (idx + 1) % 100 == 0:
        print(f"Processed {idx + 1}/{len(train_df)}")

# Combine color features and deep features
X_color = np.array(color_features_list)
X_deep = np.array(deep_features_list)
X_warna_combined = np.concatenate([X_color, X_deep], axis=1)
y_warna = np.array(labels_warna)

print(f"\nColor features shape: {X_color.shape}")
print(f"Deep features shape: {X_deep.shape}")
print(f"Combined features shape: {X_warna_combined.shape}")
print(f"Labels: {len(y_warna)}")

## 14. Train-Test Split & Scaling untuk Warna

In [ ]:
# Split data
X_train_warna, X_test_warna, y_train_warna, y_test_warna = train_test_split(
    X_warna_combined, y_warna, test_size=0.2, random_state=42, stratify=y_warna
)

# Scale features
scaler_warna = StandardScaler()
X_train_warna_scaled = scaler_warna.fit_transform(X_train_warna)
X_test_warna_scaled = scaler_warna.transform(X_test_warna)

print(f"Train set: {X_train_warna_scaled.shape}")
print(f"Test set: {X_test_warna_scaled.shape}")

## 15. Compute Class Weights untuk Warna

In [ ]:
# Compute class weights
unique_classes_warna = np.unique(y_train_warna)
class_weights_warna = compute_class_weight('balanced', classes=unique_classes_warna, y=y_train_warna)
class_weight_dict_warna = dict(zip(unique_classes_warna, class_weights_warna))
sample_weights_warna = np.array([class_weight_dict_warna[label] for label in y_train_warna])

print("Class weights untuk Warna:")
for cls, weight in class_weight_dict_warna.items():
    print(f"  {cls}: {weight:.4f}")

## 16. Train Traditional ML Models untuk Warna

In [ ]:
# Initialize models for Warna dengan lebih banyak algoritma
from sklearn.svm import SVC
from sklearn.ensemble import ExtraTreesClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
try:
    from lightgbm import LGBMClassifier
    has_lightgbm = True
except ImportError:
    has_lightgbm = False
    print("LightGBM not installed, skipping LightGBM model")

models_warna = {
    'XGBoost': XGBClassifier(n_estimators=200, max_depth=7, learning_rate=0.1, random_state=42),
    'HistGradient': HistGradientBoostingClassifier(max_iter=200, max_depth=7, random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=200, max_depth=15, class_weight='balanced', random_state=42),
    'ExtraTrees': ExtraTreesClassifier(n_estimators=200, max_depth=15, class_weight='balanced', random_state=42),
    'GradientBoosting': GradientBoostingClassifier(n_estimators=150, max_depth=7, learning_rate=0.1, random_state=42),
    'SVM': SVC(kernel='rbf', C=10, gamma='scale', class_weight='balanced', random_state=42, probability=True),
    'KNN': KNeighborsClassifier(n_neighbors=7, weights='distance', metric='minkowski'),
    'LogisticRegression': LogisticRegression(max_iter=1000, class_weight='balanced', C=1.0, random_state=42),
    'Ridge': RidgeClassifier(alpha=1.0, class_weight='balanced', random_state=42),
    'AdaBoost': AdaBoostClassifier(n_estimators=100, learning_rate=0.5, random_state=42)
}

# Add LightGBM if available
if has_lightgbm:
    models_warna['LightGBM'] = LGBMClassifier(n_estimators=200, max_depth=7, learning_rate=0.1, 
                                              class_weight='balanced', random_state=42, verbose=-1)

print(f"Total models for Warna classification: {len(models_warna)}")

# Train and evaluate individual models
results_warna = {}
for name, model in models_warna.items():
    print(f"\nTraining {name} for Warna...")
    
    if name in ['XGBoost', 'HistGradient', 'GradientBoosting', 'LightGBM']:
        try:
            model.fit(X_train_warna_scaled, y_train_warna, sample_weight=sample_weights_warna)
        except TypeError:
            # Some models might not support sample_weight
            model.fit(X_train_warna_scaled, y_train_warna)
    else:
        model.fit(X_train_warna_scaled, y_train_warna)
    
    y_pred = model.predict(X_test_warna_scaled)
    accuracy = accuracy_score(y_test_warna, y_pred)
    results_warna[name] = accuracy
    
    print(f"{name} Accuracy: {accuracy:.4f}")

print("\n" + "="*60)
print("Individual Model Results for Warna:")
sorted_results = sorted(results_warna.items(), key=lambda x: x[1], reverse=True)
for name, acc in sorted_results:
    print(f"{name}: {acc:.4f}")

## 17. Create Ensemble Model untuk Warna

In [ ]:
# Create ensemble for Warna
ensemble_warna = VotingClassifier(
    estimators=list(models_warna.items()),
    voting='hard'
)

print("Training ensemble model for Warna...")
ensemble_warna.fit(X_train_warna_scaled, y_train_warna)

y_pred_ensemble_warna = ensemble_warna.predict(X_test_warna_scaled)
ensemble_accuracy_warna = accuracy_score(y_test_warna, y_pred_ensemble_warna)

print(f"\nEnsemble Accuracy for Warna: {ensemble_accuracy_warna:.4f}")
print("\nClassification Report untuk Warna (Ensemble):")
print(classification_report(y_test_warna, y_pred_ensemble_warna))

## 18. Confusion Matrix untuk Warna

In [ ]:
cm_warna = confusion_matrix(y_test_warna, y_pred_ensemble_warna)
plt.figure(figsize=(12, 10))
sns.heatmap(cm_warna, annot=True, fmt='d', cmap='Greens',
            xticklabels=np.unique(y_warna),
            yticklabels=np.unique(y_warna))
plt.title('Confusion Matrix - Warna Classification (Multi-Feature Ensemble)')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

## 19. Save Models

In [ ]:
import pickle

# Save ensemble model untuk Jenis
with open('ensemble_jenis.pkl', 'wb') as f:
    pickle.dump(ensemble_jenis, f)

# Save ensemble model untuk Warna
with open('ensemble_warna.pkl', 'wb') as f:
    pickle.dump(ensemble_warna, f)

# Save scalers
with open('scaler_jenis.pkl', 'wb') as f:
    pickle.dump(scaler, f)

with open('scaler_warna.pkl', 'wb') as f:
    pickle.dump(scaler_warna, f)

# Save label mappings untuk Warna
unique_warna_labels = np.unique(y_warna)
warna_label_to_idx = {label: idx for idx, label in enumerate(unique_warna_labels)}
warna_idx_to_label = {idx: label for label, idx in warna_label_to_idx.items()}

with open('warna_label_mapping.pkl', 'wb') as f:
    pickle.dump({
        'label_to_idx': warna_label_to_idx,
        'idx_to_label': warna_idx_to_label
    }, f)

# Save label mappings untuk Jenis
unique_jenis_labels = np.unique(y_jenis)
jenis_label_to_idx = {label: idx for idx, label in enumerate(unique_jenis_labels)}
jenis_idx_to_label = {idx: label for label, idx in jenis_label_to_idx.items()}

with open('jenis_label_mapping.pkl', 'wb') as f:
    pickle.dump({
        'label_to_idx': jenis_label_to_idx,
        'idx_to_label': jenis_idx_to_label
    }, f)

print("Models saved successfully:")
print("  - ensemble_jenis.pkl")
print("  - ensemble_warna.pkl")
print("  - scaler_jenis.pkl")
print("  - scaler_warna.pkl")
print("  - warna_label_mapping.pkl")
print("  - jenis_label_mapping.pkl")

## 20. Load Test Data

In [ ]:
# Load sample submission
submission_df = pd.read_csv('sample_submission.csv')

print(f"Test samples: {len(submission_df)}")
print(f"\nSubmission format:")
print(submission_df.head())
print(f"\nTest IDs range: {submission_df['id'].min()} to {submission_df['id'].max()}")

## 21. Prediction Function untuk Test Data

In [ ]:
def predict_test_image(image_id, image_dir):
    """
    Predict jenis and warna for a test image using EfficientNet-B0 features
    Returns: (jenis_label, warna_label) or (None, None) if image not found
    """
    # Load image
    image, _ = load_image(image_id, image_dir)
    if image is None:
        return None, None
    
    # Preprocess and detect with OWL-ViT
    preprocessed = preprocess_image(image)
    boxes, scores = detect_object(preprocessed)
    cropped = crop_object(preprocessed, boxes)
    
    # Predict Jenis
    # Extract EfficientNet-B0 deep features for Jenis
    deep_features_jenis = extract_features(cropped)
    features_jenis_scaled = scaler.transform([deep_features_jenis])
    jenis_pred = ensemble_jenis.predict(features_jenis_scaled)[0]
    
    # Predict Warna
    # Extract multi-modal color features + EfficientNet-B0 deep features for Warna
    # color_features = extract_combined_color_features(cropped)
    deep_features_warna = extract_features(cropped)
    # combined_features = np.concatenate([color_features, deep_features_warna])
    features_warna_scaled = scaler_warna.transform([deep_features_warna])
    warna_pred = ensemble_warna.predict(features_warna_scaled)[0]
    
    return jenis_pred, warna_pred

print("Prediction function defined")
print("Pipeline: OWL-ViT → Crop → Feature Extraction → Ensemble Prediction")

## 22. Generate Predictions untuk Test Dataset

In [ ]:
print("Generating predictions for test dataset...")
print(f"Total test images: {len(submission_df)}\n")

predictions_jenis = []
predictions_warna = []
failed_images = []

for idx, row in submission_df.iterrows():
    image_id = row['id']
    
    # Predict
    jenis_pred, warna_pred = predict_test_image(image_id, test_dir)
    
    if jenis_pred is None or warna_pred is None:
        failed_images.append(image_id)
        # Use default values if prediction fails
        predictions_jenis.append(0)
        predictions_warna.append(0)
    else:
        # Convert labels to indices for submission
        jenis_idx = jenis_label_to_idx.get(jenis_pred, 0)
        warna_idx = warna_label_to_idx.get(warna_pred, 0)
        
        predictions_jenis.append(jenis_idx)
        predictions_warna.append(warna_idx)
    
    # Progress update
    if (idx + 1) % 50 == 0:
        print(f"Processed {idx + 1}/{len(submission_df)} images")

print(f"\nPrediction completed!")
print(f"Successful predictions: {len(submission_df) - len(failed_images)}")
print(f"Failed images: {len(failed_images)}")

if failed_images:
    print(f"Failed image IDs: {failed_images[:10]}...")  # Show first 10

## 23. Create Submission File

In [ ]:
# Update submission dataframe
submission_df['jenis'] = predictions_jenis
submission_df['warna'] = predictions_warna

# Save submission file
submission_file = 'submission.csv'
submission_df.to_csv(submission_file, index=False)

print(f"Submission file saved: {submission_file}")
print(f"\nSubmission preview:")
print(submission_df.head(10))

print(f"\nPrediction distribution:")
print(f"Jenis distribution:")
print(pd.Series(predictions_jenis).value_counts().sort_index())
print(f"\nWarna distribution:")
print(pd.Series(predictions_warna).value_counts().sort_index())

print(f"\nSubmission file ready for upload!")

## 24. Visualize Sample Predictions

In [ ]:
# Visualize 6 sample predictions
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

sample_indices = np.random.choice(len(submission_df), 6, replace=False)

for idx, ax in enumerate(axes):
    sample_idx = sample_indices[idx]
    image_id = submission_df.iloc[sample_idx]['id']
    pred_jenis = submission_df.iloc[sample_idx]['jenis']
    pred_warna = submission_df.iloc[sample_idx]['warna']
    
    # Load and display image
    image, _ = load_image(image_id, test_dir)
    
    if image is not None:
        # Get cropped image
        preprocessed = preprocess_image(image)
        boxes, scores = detect_object(preprocessed)
        cropped = crop_object(preprocessed, boxes)
        
        ax.imshow(cropped)
        
        # Get label names
        jenis_name = jenis_idx_to_label.get(pred_jenis, f"Unknown ({pred_jenis})")
        warna_name = warna_idx_to_label.get(pred_warna, f"Unknown ({pred_warna})")
        
        ax.set_title(f'ID: {image_id}\nJenis: {jenis_name}\nWarna: {warna_name}', 
                    fontsize=10)
    else:
        ax.text(0.5, 0.5, f'Image {image_id}\nNot Found', 
               ha='center', va='center')
    
    ax.axis('off')

plt.tight_layout()
plt.suptitle('Sample Test Predictions', fontsize=14, y=1.02)
plt.show()